# News Impact Curve: Visualizando a Assimetria

A **News Impact Curve (NIC)** foi introduzida por **Engle & Ng (1993)** como ferramenta
para visualizar como choques passados ($\varepsilon_{t-1}$) afetam a variancia condicional
corrente ($\sigma_t^2$).

**Definicao:**

A NIC plota $\sigma_t^2$ como funcao de $\varepsilon_{t-1}$, mantendo
$\sigma_{t-1}^2$ fixo em seu nivel de longo prazo $\bar{\sigma}^2$.

**Por que e importante?**
- Permite comparar **visualmente** como diferentes modelos respondem a choques
- Revela a **assimetria** (efeito alavancagem) de modelos como EGARCH, GJR e APARCH
- Um modelo GARCH simetrico produz uma NIC em forma de **U simetrico**
- Modelos assimetricos produzem NICs **deslocadas** para o lado negativo

**Neste notebook:**
1. NIC do GARCH(1,1) — simetrica
2. NIC do EGARCH — assimetrica via parametro $\gamma$
3. NIC do GJR-GARCH — assimetrica via indicadora $I(\varepsilon < 0)$
4. NIC do APARCH — assimetrica com power parameter $\delta$
5. Comparacao de todas as NICs
6. Interpretacao economica

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from archbox.models import APARCH, EGARCH, GARCH, GJRGARCH
from archbox.utils.news_impact import compare_news_impact, news_impact_curve, plot_news_impact

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns'].values

print(f"Dataset: {len(returns)} observacoes de retornos diarios S&P 500")
print(f"Desvio padrao amostral: {returns.std():.6f}")

# Estimar os 4 modelos
print("\nEstimando modelos...")
model_garch = GARCH(returns, p=1, q=1)
results_garch = model_garch.fit(disp=False)

model_egarch = EGARCH(returns, p=1, q=1)
results_egarch = model_egarch.fit(disp=False)

model_gjr = GJRGARCH(returns, p=1, q=1)
results_gjr = model_gjr.fit(disp=False)

model_aparch = APARCH(returns, p=1, q=1)
results_aparch = model_aparch.fit(disp=False)

print("Todos os modelos estimados com sucesso.")

## 1. NIC do GARCH(1,1)

A News Impact Curve avalia $\sigma_t^2$ como funcao de $\varepsilon_{t-1}$,
fixando $\sigma_{t-1}^2 = \bar{\sigma}^2$ (variancia media condicional).

Para o **GARCH(1,1)**:

$$\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \bar{\sigma}^2$$

Note que a NIC e uma **parabola simetrica** em $\varepsilon_{t-1}$:
- Depende apenas de $\varepsilon_{t-1}^2$ (magnitude, nao sinal)
- Choques positivos e negativos de mesma magnitude produzem o **mesmo impacto**
- O minimo ocorre em $\varepsilon_{t-1} = 0$

In [ ]:
# NIC analitica para GARCH(1,1): sigma2 = omega + alpha*eps^2 + beta*sigma2_bar

# Usando a funcao news_impact_curve do archbox
eps_garch, sigma2_garch = news_impact_curve(model_garch, results_garch, n_points=200)

# Plotar
ax = plot_news_impact(eps_garch, sigma2_garch, model_name='GARCH(1,1)')
ax.set_title('News Impact Curve: GARCH(1,1) — Simetrica')

# Anotar a simetria
ax.annotate('Simetrica:\nmesmo impacto\npara + e -',
            xy=(eps_garch[30], sigma2_garch[30]),
            xytext=(eps_garch[20], sigma2_garch[10]),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.show()

# Parametros estimados
print("Parametros do GARCH(1,1):")
for name, val in zip(results_garch.param_names, results_garch.params, strict=False):
    print(f"  {name}: {val:.6f}")

## 2. NIC do EGARCH

O **EGARCH** (Nelson, 1991) modela o **logaritmo** da variancia condicional:

$$\ln(\sigma_t^2) = \omega + \alpha(|z_{t-1}| - \sqrt{2/\pi}) + \gamma z_{t-1} + \beta \ln(\sigma_{t-1}^2)$$

A assimetria e capturada pelo parametro $\gamma$:
- $\gamma < 0$: choques negativos aumentam **mais** a volatilidade (efeito alavancagem)
- $\gamma = 0$: sem assimetria (equivale ao GARCH simetrico em escala log)
- $\gamma > 0$: choques positivos aumentam mais a volatilidade (raro)

A NIC do EGARCH e **exponencial e assimetrica**: mais ingreme para choques negativos.

In [ ]:
# NIC do EGARCH: log(sigma2) = omega + alpha*|z| + gamma*z + beta*log(sigma2_bar)

eps_egarch, sigma2_egarch = news_impact_curve(model_egarch, results_egarch, n_points=200)

ax = plot_news_impact(eps_egarch, sigma2_egarch, model_name='EGARCH(1,1)')
ax.set_title('News Impact Curve: EGARCH(1,1) — Assimetrica Exponencial')

plt.show()

# Parametros estimados
print("Parametros do EGARCH(1,1):")
for name, val in zip(results_egarch.param_names, results_egarch.params, strict=False):
    print(f"  {name}: {val:.6f}")

# Verificar assimetria
gamma_idx = [i for i, n in enumerate(results_egarch.param_names) if 'gamma' in n.lower()]
if gamma_idx:
    gamma_val = results_egarch.params[gamma_idx[0]]
    print(f"\nParametro gamma = {gamma_val:.6f}")
    if gamma_val < 0:
        print("gamma < 0 → efeito alavancagem: choques negativos aumentam mais a volatilidade")
    else:
        print("gamma >= 0 → sem efeito alavancagem tipico")

## 3. NIC do GJR-GARCH

O **GJR-GARCH** (Glosten, Jagannathan & Runkle, 1993) usa uma **funcao indicadora**
para capturar a assimetria:

$$\sigma_t^2 = \omega + (\alpha + \gamma \cdot \mathbf{1}(\varepsilon_{t-1} < 0)) \cdot \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

A NIC e uma **parabola com inclinacao diferente** para cada lado:
- Para $\varepsilon_{t-1} \geq 0$: inclinacao $= \alpha$
- Para $\varepsilon_{t-1} < 0$: inclinacao $= \alpha + \gamma$

Se $\gamma > 0$: choques negativos tem impacto **maior** que positivos.

In [ ]:
# NIC do GJR-GARCH: sigma2 = omega + (alpha + gamma*I(eps<0))*eps^2 + beta*sigma2_bar

eps_gjr, sigma2_gjr = news_impact_curve(model_gjr, results_gjr, n_points=200)

ax = plot_news_impact(eps_gjr, sigma2_gjr, model_name='GJR-GARCH(1,1)')
ax.set_title('News Impact Curve: GJR-GARCH(1,1) — Parabola Assimetrica')

plt.show()

# Parametros estimados
print("Parametros do GJR-GARCH(1,1):")
for name, val in zip(results_gjr.param_names, results_gjr.params, strict=False):
    print(f"  {name}: {val:.6f}")

print("\nNote a 'quebra' na inclinacao em eps=0:")
print("A parabola e mais ingreme para choques negativos (lado esquerdo).")

## 4. NIC do APARCH

O **APARCH** (Ding, Granger & Engle, 1993) generaliza varios modelos com um **power parameter** $\delta$:

$$\sigma_t^\delta = \omega + \alpha(|\varepsilon_{t-1}| - \gamma \varepsilon_{t-1})^\delta + \beta \sigma_{t-1}^\delta$$

Casos especiais:
- $\delta = 2, \gamma = 0$: GARCH padrao
- $\delta = 2, \gamma \neq 0$: GJR-GARCH
- $\delta = 1$: Taylor/Schwert GARCH (modela $\sigma_t$ diretamente)

A NIC do APARCH e uma **curva de potencia assimetrica** — a forma depende de $\delta$.

In [ ]:
# NIC do APARCH: sigma^delta = omega + (alpha*(|eps|-gamma*eps))^delta + beta*sigma^delta_bar

eps_aparch, sigma2_aparch = news_impact_curve(model_aparch, results_aparch, n_points=200)

ax = plot_news_impact(eps_aparch, sigma2_aparch, model_name='APARCH(1,1)')
ax.set_title('News Impact Curve: APARCH(1,1) — Power Assimetrica')

plt.show()

# Parametros estimados
print("Parametros do APARCH(1,1):")
for name, val in zip(results_aparch.param_names, results_aparch.params, strict=False):
    print(f"  {name}: {val:.6f}")

# Identificar delta
delta_idx = [i for i, n in enumerate(results_aparch.param_names) if 'delta' in n.lower()]
if delta_idx:
    delta_val = results_aparch.params[delta_idx[0]]
    print(f"\nPower parameter delta = {delta_val:.4f}")
    if abs(delta_val - 2.0) < 0.1:
        print("delta ~= 2 → proximo do GARCH/GJR padrao")
    elif abs(delta_val - 1.0) < 0.1:
        print("delta ~= 1 → modela sigma_t diretamente (Taylor/Schwert)")
    else:
        print(f"delta = {delta_val:.4f} → potencia otima estimada pelos dados")

## 5. Comparacao de NICs

Agora plotamos as **quatro NICs no mesmo grafico** para comparacao direta.

Esperamos ver:
- **GARCH**: parabola simetrica
- **EGARCH**: curva exponencial assimetrica (mais ingreme a esquerda)
- **GJR-GARCH**: parabola com "quebra" em $\varepsilon = 0$
- **APARCH**: curva de potencia assimetrica

Todas as curvas compartilham o eixo $\varepsilon_{t-1}$ (eixo x)
e mostram $\sigma_t^2$ (eixo y).

In [ ]:
# Plot comparativo: 4 NICs no mesmo grafico com cores diferentes

models_results = [
    (model_garch, results_garch, 'GARCH(1,1)'),
    (model_egarch, results_egarch, 'EGARCH(1,1)'),
    (model_gjr, results_gjr, 'GJR-GARCH(1,1)'),
    (model_aparch, results_aparch, 'APARCH(1,1)'),
]

ax = compare_news_impact(models_results, n_points=200, sigma_range=3.0)
ax.set_title('Comparacao de News Impact Curves', fontsize=13)
ax.set_xlabel(r'$\varepsilon_{t-1}$ (choque)')
ax.set_ylabel(r'$\sigma_t^2$ (variancia condicional)')

plt.tight_layout()
plt.show()

print("Observacoes:")
print("- GARCH: simetrica (parabola centrada)")
print("- EGARCH: mais ingreme para choques negativos (exponencial)")
print("- GJR-GARCH: inclinacao diferente a esquerda e direita")
print("- APARCH: forma intermediaria controlada pelo power parameter")

## 6. Interpretacao economica e razao de assimetria

A assimetria na volatilidade tem uma interpretacao economica clara:

**Efeito alavancagem** (Black, 1976; Christie, 1982):
- Quando o preco de uma acao cai ($\varepsilon_t < 0$), a razao divida/equity aumenta
- A empresa se torna mais alavancada → mais arriscada → volatilidade sobe
- Quando o preco sobe ($\varepsilon_t > 0$), o efeito e oposto (menor volatilidade)

**Quantificacao:** podemos medir a **razao de impacto negativo/positivo**:

$$\text{Razao de assimetria} = \frac{\sigma^2(\varepsilon = -k)}{\sigma^2(\varepsilon = +k)}$$

Se $> 1$: choques negativos tem impacto maior (efeito alavancagem).
Se $= 1$: modelo simetrico.

In [ ]:
# Razao de impacto negativo/positivo para cada modelo: nic(-1)/nic(+1)

def compute_asymmetry_ratio(model, results, k_sigmas=2.0):
    """Compute ratio of negative to positive shock impact.

    Uses shocks of +/- k_sigmas * unconditional_sigma.
    """
    sigma_ref = results.conditional_volatility.mean()
    k = k_sigmas * sigma_ref

    sigma2_ref = sigma_ref ** 2

    # Impact of negative shock
    sigma2_neg = model._one_step_variance(-k, sigma2_ref, results.params)
    # Impact of positive shock
    sigma2_pos = model._one_step_variance(+k, sigma2_ref, results.params)

    return sigma2_neg / sigma2_pos if sigma2_pos > 0 else np.nan

print("Razao de Assimetria (impacto negativo / impacto positivo)")
print("Choque = +/- 2 sigma incondicionais")
print("=" * 55)
print(f"{'Modelo':<20} {'Razao':>10} {'Interpretacao':>20}")
print("-" * 55)

models_info = [
    ('GARCH(1,1)', model_garch, results_garch),
    ('EGARCH(1,1)', model_egarch, results_egarch),
    ('GJR-GARCH(1,1)', model_gjr, results_gjr),
    ('APARCH(1,1)', model_aparch, results_aparch),
]

for name, model, results in models_info:
    ratio = compute_asymmetry_ratio(model, results, k_sigmas=2.0)
    if abs(ratio - 1.0) < 0.01:
        interp = "Simetrico"
    elif ratio > 1.0:
        interp = "Alavancagem"
    else:
        interp = "Assimetria inversa"
    print(f"{name:<20} {ratio:>10.4f} {interp:>20}")

print("\nRazao = 1.0 → simetrico")
print("Razao > 1.0 → choques negativos tem impacto maior (efeito alavancagem)")
print("Quanto maior a razao, mais forte a assimetria.")

In [ ]:
# Visualizacao da razao de assimetria para diferentes magnitudes de choque

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel esquerdo: razao em funcao da magnitude do choque
k_range = np.linspace(0.5, 4.0, 50)
colors = ['steelblue', 'darkorange', 'green', 'red']

for (name, model, res), color in zip(models_info, colors, strict=False):
    ratios = [compute_asymmetry_ratio(model, res, k_sigmas=k) for k in k_range]
    axes[0].plot(k_range, ratios, label=name, color=color, linewidth=2)

axes[0].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Simetria')
axes[0].set_xlabel('Magnitude do choque (multiplos de $\\sigma$)')
axes[0].set_ylabel('Razao negativo/positivo')
axes[0].set_title('Razao de Assimetria vs Magnitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Painel direito: barplot da razao para k=2
model_names = [name for name, _, _ in models_info]
ratios_2sigma = [compute_asymmetry_ratio(model, res, k_sigmas=2.0)
                 for _, model, res in models_info]
bar_colors = ['steelblue' if abs(r - 1.0) < 0.01 else 'darkorange' for r in ratios_2sigma]
axes[1].bar(model_names, ratios_2sigma, color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Razao negativo/positivo')
axes[1].set_title('Razao de Assimetria (choque = 2$\\sigma$)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("O GARCH simetrico tem razao = 1.0 para qualquer magnitude.")
print("Modelos assimetricos mostram razao > 1.0, especialmente para choques grandes.")